In [28]:
# Install necessary libraries (nltk is usually pre-installed, but good practice)
# !pip install nltk scikit-learn pandas

import pandas as pd
import numpy as np
import re
import string
import nltk # Make sure nltk is imported before using nltk.download

# --- Download NLTK data ---
# We need to catch LookupError when the resource is not found

# List of required NLTK resources
resources = ['wordnet', 'stopwords', 'punkt', 'punkt_tab'] # Added 'punkt_tab'

for resource in resources:
    try:
        # Construct the path NLTK uses internally to check
        if resource in ['wordnet', 'stopwords']:
             path_check = f'corpora/{resource}'
        elif resource in ['punkt', 'punkt_tab']: # punkt_tab is also a tokenizer resource
             path_check = f'tokenizers/{resource}'
        else:
             path_check = resource # Default if unsure, might need adjustment

        nltk.data.find(path_check)
        print(f"NLTK resource '{resource}' found.")
    except LookupError:
        print(f"NLTK resource '{resource}' not found. Downloading...")
        nltk.download(resource, quiet=True) # Download if not found
        print(f"'{resource}' downloaded.")

# --- Now import the rest of the modules that DEPEND on the downloaded data ---
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

from sklearn.model_selection import train_test_split, LeaveOneGroupOut, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.base import clone # Needed for cloning pipeline in CV loop

print("\nSetup Complete.")

NLTK resource 'wordnet' not found. Downloading...
'wordnet' downloaded.
NLTK resource 'stopwords' found.
NLTK resource 'punkt' found.
NLTK resource 'punkt_tab' found.

Setup Complete.


In [29]:
from google.colab import files

print("Please upload the 'data_stories_one_shot.csv' file:")
uploaded = files.upload()

# Get the filename (it should be 'data_stories_one_shot.csv')
try:
    filename = next(iter(uploaded))
    print(f"\nSuccessfully uploaded '{filename}'")
except StopIteration:
    print("\nNo file was uploaded!")
    filename = None # Set filename to None if upload failed

Please upload the 'data_stories_one_shot.csv' file:


Saving data_stories_one_shot.csv to data_stories_one_shot (2).csv

Successfully uploaded 'data_stories_one_shot (2).csv'


In [30]:
# --- Load Data ---
if filename: # Only proceed if a file was uploaded
    print(f"Loading data from '{filename}'...")
    df = pd.read_csv(filename)
    print(f"Data loaded with {df.shape[0]} rows and {df.shape[1]} columns.")
    print("\nFirst 5 rows:")
    print(df.head())
    print("\nValue counts for 'Stage':")
    print(df['Stage'].value_counts())
    data_loaded = True
else:
    print("Cannot load data, file upload failed in the previous step.")
    data_loaded = False

Loading data from 'data_stories_one_shot (2).csv'...
Data loaded with 130 rows and 4 columns.

First 5 rows:
  Plot_Name  Stage  Quality                                           Sentence
0  walk dog      1      1.0              This is a line chart with error bars.
1  walk dog      1      1.0                     The chart title is 'Walk dog'.
2  walk dog      1      1.0              The y-axis represents 'Mean anxiety'.
3  walk dog      1      1.0  The x-axis indicates conditions such as 'Basel...
4  walk dog      1      1.0  The chart compares mean anxiety levels with an...

Value counts for 'Stage':
Stage
1    79
2    48
3     3
Name: count, dtype: int64


In [31]:
# --- Define Target Variable ---
if data_loaded:
    # Show = Level 1 (encode as 0)
    # Tell = Level 2 and 3 (encode as 1)
    df['label'] = df['Stage'].apply(lambda x: 0 if x == 1 else 1)
    print("\nValue counts for new 'label' (0=Show, 1=Tell):")
    print(df['label'].value_counts())


Value counts for new 'label' (0=Show, 1=Tell):
label
0    79
1    51
Name: count, dtype: int64


In [32]:
# --- Text Preprocessing Setup ---
if data_loaded:
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()

    def preprocess_text(text):
        if pd.isna(text):
            return ""
        # 1. Lowercase
        text = text.lower()
        # 2. Remove punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))
        # 3. Tokenization
        tokens = word_tokenize(text)
        # 4. Remove stop words
        tokens = [word for word in tokens if word not in stop_words]
        # 5. Lemmatization
        tokens = [lemmatizer.lemmatize(word) for word in tokens]
        # Join back into string for vectorizer
        return ' '.join(tokens)

    # --- Apply Preprocessing ---
    print("\nPreprocessing text...")
    df['processed_sentence'] = df['Sentence'].apply(preprocess_text)
    print("Preprocessing complete.")
    print("\nSample processed sentences:")
    print(df[['Sentence', 'processed_sentence', 'label']].head())


Preprocessing text...
Preprocessing complete.

Sample processed sentences:
                                            Sentence  \
0              This is a line chart with error bars.   
1                     The chart title is 'Walk dog'.   
2              The y-axis represents 'Mean anxiety'.   
3  The x-axis indicates conditions such as 'Basel...   
4  The chart compares mean anxiety levels with an...   

                                  processed_sentence  label  
0                               line chart error bar      0  
1                               chart title walk dog      0  
2                      yaxis represents mean anxiety      0  
3  xaxis indicates condition baseline 30 minute p...      0  
4  chart compare mean anxiety level without dog time      0  


In [33]:
# --- Prepare Data for Models ---
if data_loaded:
    X = df['processed_sentence']
    y = df['label']
    groups = df['Plot_Name'] # For LeaveOneGroupOut CV
    print("\nData prepared for modeling (X, y, groups).")


Data prepared for modeling (X, y, groups).


In [34]:
from sklearn.ensemble import RandomForestClassifier

# --- Define Models ---
if data_loaded:
    # We use a pipeline to combine TF-IDF and the classifier
    # This prevents data leakage during cross-validation
    models = {
        "Logistic Regression": Pipeline([
            ('tfidf', TfidfVectorizer()),
            # liblinear is often good for smaller datasets and binary classification
            ('clf', LogisticRegression(solver='liblinear', random_state=42))
        ]),
        "SVM": Pipeline([
            ('tfidf', TfidfVectorizer()),
            # probability=True is needed to calculate AUC
            ('clf', SVC(probability=True, random_state=42))
        ]),
        "Naive Bayes": Pipeline([
            ('tfidf', TfidfVectorizer()),
            ('clf', MultinomialNB())
        ]),
        "Random Forest": Pipeline([
            ('tfidf', TfidfVectorizer()),
            # You can tweak the number of estimators and other hyperparameters
            ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
        ])
    }

    print("\nModels defined:")
    for name in models:
        print(f"- {name}")



Models defined:
- Logistic Regression
- SVM
- Naive Bayes
- Random Forest


In [35]:
# --- Evaluation Strategy 1: Train/Validate Split (like "Student Generated Stories" in Fig 6) ---
if data_loaded:
    print("\n--- Evaluating with Train/Validate Split (80/20) ---")
    # Stratify ensures proportion of labels is same in train and test sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(f"Train set size: {len(X_train)}, Test set size: {len(X_test)}")

    results_tv = {}
    for name, model_pipeline in models.items():
        print(f"\nTraining and evaluating {name}...")
        # Fit pipeline (TF-IDF fit_transform on train, transform on test happens internally)
        model_pipeline.fit(X_train, y_train)

        # Predict probabilities for the positive class (Tell=1)
        # Needed for AUC calculation
        y_pred_proba = model_pipeline.predict_proba(X_test)[:, 1]

        # Calculate AUC
        auc = roc_auc_score(y_test, y_pred_proba)
        results_tv[name] = auc
        print(f"{name} AUC: {auc:.4f}")

        # Optional: Print classification report for more details
        # y_pred = model_pipeline.predict(X_test)
        # print(classification_report(y_test, y_pred, target_names=['Show (0)', 'Tell (1)']))

    print("\n--- Train/Validate Split Results (AUC) ---")
    for name, score in results_tv.items():
        print(f"{name}: {score:.4f}")


--- Evaluating with Train/Validate Split (80/20) ---
Train set size: 104, Test set size: 26

Training and evaluating Logistic Regression...
Logistic Regression AUC: 0.9187

Training and evaluating SVM...
SVM AUC: 0.8750

Training and evaluating Naive Bayes...
Naive Bayes AUC: 0.8750

Training and evaluating Random Forest...
Random Forest AUC: 0.9375

--- Train/Validate Split Results (AUC) ---
Logistic Regression: 0.9187
SVM: 0.8750
Naive Bayes: 0.8750
Random Forest: 0.9375


In [36]:
# --- Evaluation Strategy 2: Leave-One-Plot-Out Cross-Validation (LOPO CV) ---
if data_loaded:
    print("\n--- Evaluating with Leave-One-Plot-Out Cross-Validation ---")
    logo = LeaveOneGroupOut()
    n_splits = logo.get_n_splits(X, y, groups)
    print(f"Number of splits for LOPO CV: {n_splits} (based on unique Plot_Name values)")

    results_lopo = {}
    for name, model_pipeline in models.items():
        print(f"\nEvaluating {name} with LOPO CV...")
        # We need to collect predictions across all folds to calculate a single overall AUC
        all_y_true = []
        all_y_pred_proba = []
        fold_aucs = [] # Store AUC per fold just for informational purposes

        for i, (train_index, test_index) in enumerate(logo.split(X, y, groups)):
            X_train_fold, X_test_fold = X.iloc[train_index], X.iloc[test_index]
            y_train_fold, y_test_fold = y.iloc[train_index], y.iloc[test_index]
            current_plot = groups.iloc[test_index].unique()[0] # Get the plot name for this fold

            # Clone the pipeline to ensure it's fresh for each fold (esp. TF-IDF)
            current_pipeline = clone(model_pipeline)

            # Fit on the training fold
            current_pipeline.fit(X_train_fold, y_train_fold)

            # Predict probabilities on the test fold
            # Handle cases where a test fold might only have one class (predict_proba needs >1 class present during training)
            # roc_auc_score also needs >1 class in y_true (y_test_fold)
            if len(np.unique(y_test_fold)) > 1:
                try:
                    y_pred_proba_fold = current_pipeline.predict_proba(X_test_fold)[:, 1]
                    fold_auc = roc_auc_score(y_test_fold, y_pred_proba_fold)
                    fold_aucs.append(fold_auc)
                    # Store results for overall AUC calculation
                    all_y_true.extend(y_test_fold)
                    all_y_pred_proba.extend(y_pred_proba_fold)
                    # print(f"  Fold {i+1}/{n_splits}, Plot: {current_plot}, Test size: {len(test_index)}, AUC: {fold_auc:.4f}")
                except ValueError as e:
                    # This might happen if predict_proba fails for some reason, though less likely with pipeline cloning
                     print(f"  Skipping Fold {i+1}/{n_splits} ({current_plot}) for {name} due to prediction error: {e}")
            else:
                # If only one class in test fold, we can't calculate AUC for this fold,
                # but we still need the predictions if the model could make them,
                # to contribute to the *overall* AUC calculation later.
                print(f"  Fold {i+1}/{n_splits} ({current_plot}) has only one class in test set - cannot calculate fold AUC.")
                # Try to get predictions anyway, but handle potential errors if predict_proba fails
                try:
                    y_pred_proba_fold = current_pipeline.predict_proba(X_test_fold)[:, 1]
                    all_y_true.extend(y_test_fold)
                    all_y_pred_proba.extend(y_pred_proba_fold)
                except ValueError as e:
                    print(f"    Could not get predictions for fold {i+1} ({current_plot}): {e}")


        # Calculate overall AUC from all collected predictions IF possible
        if len(all_y_true) > 0 and len(np.unique(all_y_true)) > 1:
            overall_auc = roc_auc_score(all_y_true, all_y_pred_proba)
            results_lopo[name] = overall_auc
            print(f"{name} Overall LOPO AUC (calculated across all folds): {overall_auc:.4f}")
            # print(f"{name} Mean Fold AUC: {np.mean(fold_aucs):.4f} (Std: {np.std(fold_aucs):.4f})") # Info only - based only on folds where AUC could be calculated
        else:
             results_lopo[name] = np.nan
             print(f"{name} Could not calculate Overall LOPO AUC (Insufficient valid predictions or only one class across all test folds).")

    print("\n--- Leave-One-Plot-Out CV Results (Overall AUC) ---")
    for name, score in results_lopo.items():
        print(f"{name}: {score:.4f}")


--- Evaluating with Leave-One-Plot-Out Cross-Validation ---
Number of splits for LOPO CV: 12 (based on unique Plot_Name values)

Evaluating Logistic Regression with LOPO CV...
  Fold 6/12 (time use) has only one class in test set - cannot calculate fold AUC.
Logistic Regression Overall LOPO AUC (calculated across all folds): 0.8719

Evaluating SVM with LOPO CV...
  Fold 6/12 (time use) has only one class in test set - cannot calculate fold AUC.
SVM Overall LOPO AUC (calculated across all folds): 0.9069

Evaluating Naive Bayes with LOPO CV...
  Fold 6/12 (time use) has only one class in test set - cannot calculate fold AUC.
Naive Bayes Overall LOPO AUC (calculated across all folds): 0.8970

Evaluating Random Forest with LOPO CV...
  Fold 6/12 (time use) has only one class in test set - cannot calculate fold AUC.
Random Forest Overall LOPO AUC (calculated across all folds): 0.8295

--- Leave-One-Plot-Out CV Results (Overall AUC) ---
Logistic Regression: 0.8719
SVM: 0.9069
Naive Bayes: 0

In [37]:
# --- Comparison with Figure 6 ---
if data_loaded:
    print("\n--- Comparison with Figure 6 (from paper) ---")
    # Note: Figure 6 shows results for specific dataset variants (LLM Generated, Student Generated)
    # and evaluation methods (Zero Shot, One Shot, Two Shot, Cross Validation, LOPO, Train/Validate).
    # We are comparing our results on 'data_stories_one_shot.csv' to the relevant lines.

    # Comparing to "Student Generated Stories - Train/Validate" (row 10-12 in Fig 6)
    print("\nFigure 6 Train/Validate AUC (Student Gen. Stories - approx): LR=0.78, NB=0.77, SVM=0.77")
    print("Our Train/Validate Split AUC (on data_stories_one_shot.csv):")
    if 'results_tv' in locals():
      for name, score in results_tv.items():
          print(f"  {name}: {score:.4f}")
    else:
      print("  Train/Validate results not available.")

    # Comparing to "LLM Generated Stories - Leave One Plot Out" (row 7-9 in Fig 6)
    # This might be the closest comparison for our LOPO CV, although the dataset source differs.
    print("\nFigure 6 LOPO CV AUC (LLM Gen. Stories - approx): LR=0.95, NB=0.95, SVM=0.95")
    print("Our LOPO CV AUC (on data_stories_one_shot.csv):")
    if 'results_lopo' in locals():
      for name, score in results_lopo.items():
          print(f"  {name}: {score:.4f}")
    else:
      print("  LOPO CV results not available.")


--- Comparison with Figure 6 (from paper) ---

Figure 6 Train/Validate AUC (Student Gen. Stories - approx): LR=0.78, NB=0.77, SVM=0.77
Our Train/Validate Split AUC (on data_stories_one_shot.csv):
  Logistic Regression: 0.9187
  SVM: 0.8750
  Naive Bayes: 0.8750
  Random Forest: 0.9375

Figure 6 LOPO CV AUC (LLM Gen. Stories - approx): LR=0.95, NB=0.95, SVM=0.95
Our LOPO CV AUC (on data_stories_one_shot.csv):
  Logistic Regression: 0.8719
  SVM: 0.9069
  Naive Bayes: 0.8970
  Random Forest: 0.8295


In [38]:
# BONUS : Sentence-BERT Embeddings + Classifiers

# Install required package
!pip install -q sentence-transformers

# Import libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import LabelEncoder
from sentence_transformers import SentenceTransformer

# Load data
df = pd.read_csv('data_stories_one_shot.csv')

# Use the actual column names from your file
TEXT_COL = 'Sentence'
LABEL_COL = 'Stage'

# Extract text and label
sentences = df[TEXT_COL].astype(str).tolist()
labels = LabelEncoder().fit_transform(df[LABEL_COL])

# Load Sentence-BERT model and get embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
sentence_embeddings = model.encode(sentences)

# Define classification models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "SVM": SVC(),
    "Random Forest": RandomForestClassifier(n_estimators=100),
    "Naive Bayes": GaussianNB()
}

# 5-fold stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Evaluate models
for name, clf in models.items():
    if name == "Naive Bayes":
        X = np.array(sentence_embeddings)  # Ensure dense input
    else:
        X = sentence_embeddings
    scores = cross_val_score(clf, X, labels, cv=cv, scoring='accuracy')
    print(f"{name:<20} Accuracy: {scores.mean():.4f} ± {scores.std():.4f}")


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


Logistic Regression  Accuracy: 0.8846 ± 0.0544
SVM                  Accuracy: 0.9154 ± 0.0377


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


Random Forest        Accuracy: 0.8692 ± 0.0188
Naive Bayes          Accuracy: 0.8538 ± 0.0890


/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
